# Notebook 1: Sound as Data

In Notebook 0 you saw that real birdsong looks nothing like a simple sine wave. But what IS that messy squiggle? How do scientists make sense of it?

This notebook takes **one bird, one recording** and builds your understanding from the ground up:

1. What is digital audio? (Spoiler: just a list of numbers)
2. What does sample rate mean?
3. How do you read a waveform?
4. What is a spectrogram and why does it matter?
5. How can you manipulate audio with code?

By the end, you'll be able to look at a spectrogram and *read* what a bird is doing.

### Save Your Own Copy

1. **File → Save a copy in Drive**
2. Rename it with your name (e.g. `YourName_Notebook_1.ipynb`)

---
## 1. Setup

Install and import the tools we need, then download the birdsong dataset.

In [ ]:
!pip install -q librosa

import os
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
from IPython.display import Audio, display

# Download the birdsong dataset
REPO_DIR = "/content/birdsong"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/williamedwardhahn/birdsong.git {REPO_DIR}
    print("Dataset downloaded!")
else:
    print("Dataset already downloaded.")

print("Setup complete!")

---
## 2. What Is Digital Audio?

Sound is air pressure changing over time. When you speak or a bird sings, it creates waves of high and low pressure that travel through the air to your ear.

A **microphone** measures that pressure thousands of times per second and writes down each measurement as a number. That’s all a WAV file is — **a long list of numbers**.

Let's load one recording and look at the raw numbers.

In [ ]:
# Load one Zebra Finch recording
zebra_finch_dir = os.path.join(REPO_DIR, "zebra_finch")
wav_file = sorted(os.listdir(zebra_finch_dir))[0]
file_path = os.path.join(zebra_finch_dir, wav_file)

# librosa.load returns two things: the audio data and the sample rate
audio, sr = librosa.load(file_path, sr=None)

print(f"File: {wav_file}")
print(f"Type of 'audio': {type(audio)}")
print(f"Number of samples: {len(audio):,}")
print(f"\nThe first 20 numbers in the file:")
print(audio[:20])
print(f"\nThat's it. Sound is just numbers.")

---
## 3. Sample Rate — How Fast Were the Numbers Recorded?

The **sample rate** tells you how many times per second the microphone measured the air pressure. It's measured in **Hz** (hertz = times per second).

- A sample rate of 22,050 Hz means 22,050 measurements per second
- CD quality is 44,100 Hz
- The higher the sample rate, the more detail you capture

The sample rate also tells the computer how fast to play back the numbers. Watch what happens when we play the same numbers at different speeds.

In [ ]:
print(f"Original sample rate: {sr} Hz")
print(f"Duration: {len(audio)/sr:.2f} seconds")
print(f"\n--- Normal speed ---")
display(Audio(audio, rate=sr))

In [ ]:
# Play at HALF the sample rate — sounds slow and low-pitched
print("--- Half speed (sounds low and slow) ---")
display(Audio(audio, rate=sr // 2))

In [ ]:
# Play at DOUBLE the sample rate — sounds fast and high-pitched
print("--- Double speed (sounds high and fast) ---")
display(Audio(audio, rate=sr * 2))

The numbers didn’t change — we just told the computer to play them back at a different speed. That’s all sample rate is.

---
## 4. Waveform — Drawing the Numbers as a Line

A **waveform** is just a plot of those numbers. Time goes left to right, and the air pressure value goes up and down. It’s the same list of numbers from Section 2, drawn as a line.

In [ ]:
plt.figure(figsize=(12, 3))
librosa.display.waveshow(audio, sr=sr)
plt.title(f"Waveform — {wav_file}")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude (air pressure)")
plt.show()

print("Tall peaks = loud moments. Flat sections = silence.")
print("You can see the bird's song bursts, but you can't tell what notes it's singing.")
print("For that, we need a spectrogram.")

---
## 5. What Is a Spectrogram?

A waveform shows *when* sounds happen, but not *what frequencies* are present. A bird might be singing a high note and a low note at the same time — the waveform just shows you the combined squiggle.

A **spectrogram** pulls the frequencies apart. It’s a picture where:
- **X-axis** = time (left to right)
- **Y-axis** = frequency / pitch (low at bottom, high at top)
- **Color** = loudness (brighter = louder)

Let’s build intuition step by step.

### Step 1: A single pure tone (440 Hz)

A pure tone has just one frequency. Its spectrogram should be a single horizontal line.

In [ ]:
# Generate a pure 440 Hz tone (1 second)
duration = 1.0
t = np.linspace(0, duration, int(sr * duration), endpoint=False)
tone_440 = np.sin(2 * np.pi * 440 * t).astype(np.float32)

fig, axes = plt.subplots(1, 2, figsize=(14, 3))

# Waveform
axes[0].plot(t[:500], tone_440[:500], color='#2ecc71')
axes[0].set_title("Waveform: 440 Hz Tone")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")

# Spectrogram
S = np.abs(librosa.stft(tone_440))
S_db = librosa.amplitude_to_db(S, ref=np.max)
librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='linear', ax=axes[1])
axes[1].set_title("Spectrogram: 440 Hz Tone")
axes[1].set_ylim(0, 2000)

plt.tight_layout()
plt.show()

print("One frequency = one horizontal line on the spectrogram.")
display(Audio(tone_440, rate=sr))

### Step 2: Two tones at once (440 Hz + 880 Hz)

If we add a second frequency, the waveform gets messy — but the spectrogram clearly shows both.

In [ ]:
# Two tones played at the same time
tone_880 = np.sin(2 * np.pi * 880 * t).astype(np.float32)
two_tones = (tone_440 + tone_880) / 2  # Mix them together

fig, axes = plt.subplots(1, 2, figsize=(14, 3))

# Waveform — looks complicated!
axes[0].plot(t[:500], two_tones[:500], color='#9b59b6')
axes[0].set_title("Waveform: Two Tones Combined")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")

# Spectrogram — clearly shows both frequencies
S = np.abs(librosa.stft(two_tones))
S_db = librosa.amplitude_to_db(S, ref=np.max)
librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='linear', ax=axes[1])
axes[1].set_title("Spectrogram: Two Tones (440 + 880 Hz)")
axes[1].set_ylim(0, 2000)

plt.tight_layout()
plt.show()

print("The waveform is a mess, but the spectrogram PULLS THE NOTES APART.")
print("Two frequencies = two horizontal lines.")
display(Audio(two_tones, rate=sr))

### Step 3: A chirp (frequency sweeping up)

Birds don't sing steady tones — they sweep between frequencies. Let's make a sound that starts low and goes high, like a whistle sliding up.

In [ ]:
# A chirp that sweeps from 300 Hz to 3000 Hz over 1 second
freq_start = 300
freq_end = 3000
frequencies = np.linspace(freq_start, freq_end, len(t))
# Instantaneous phase from cumulative frequency
phase = 2 * np.pi * np.cumsum(frequencies) / sr
chirp = np.sin(phase).astype(np.float32)

fig, axes = plt.subplots(1, 2, figsize=(14, 3))

# Waveform
axes[0].plot(t, chirp, color='#e67e22', linewidth=0.5)
axes[0].set_title("Waveform: Chirp (300→3000 Hz)")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")

# Spectrogram — a diagonal line!
S = np.abs(librosa.stft(chirp))
S_db = librosa.amplitude_to_db(S, ref=np.max)
librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='linear', ax=axes[1])
axes[1].set_title("Spectrogram: Chirp (diagonal = rising pitch)")
axes[1].set_ylim(0, 4000)

plt.tight_layout()
plt.show()

print("A rising pitch = a diagonal line on the spectrogram.")
print("Now you can READ what the sound is doing just by looking at the picture.")
display(Audio(chirp, rate=sr))

### Step 4: Now look at the REAL birdsong

With what you just learned, you can read this! Look for:
- Horizontal lines = steady notes
- Diagonal lines = sweeping pitch
- Gaps = silence between song elements
- Bright spots = loud moments

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# Waveform
librosa.display.waveshow(audio, sr=sr, ax=axes[0])
axes[0].set_title(f"Waveform — {wav_file}")
axes[0].set_xlabel("")

# Spectrogram
S = np.abs(librosa.stft(audio))
S_db = librosa.amplitude_to_db(S, ref=np.max)
librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='log', ax=axes[1])
axes[1].set_title(f"Spectrogram — {wav_file}")

plt.tight_layout()
plt.show()

print("Now you can READ this!")
print("The bird is sweeping between frequencies, trilling, pausing.")
print("Each bright trace is a note or syllable in the bird's song.")
print("\nListen again and follow along on the spectrogram:")
display(Audio(audio, rate=sr))

---
## 6. Manipulate the Audio

You’re not just looking at data — you can *change* it. Since audio is just a list of numbers, you can rearrange, scale, or modify those numbers and hear the result.

In [ ]:
# --- Reverse it ---
reversed_audio = audio[::-1]  # Read the list backwards

plt.figure(figsize=(12, 3))
librosa.display.waveshow(reversed_audio, sr=sr)
plt.title("Reversed Birdsong")
plt.show()

print("Reversed — the song plays backwards:")
display(Audio(reversed_audio, rate=sr))

In [ ]:
# --- Speed it up (take every other sample) ---
fast_audio = audio[::2]  # Skip every other number

plt.figure(figsize=(12, 3))
librosa.display.waveshow(fast_audio, sr=sr)
plt.title("Sped Up (every other sample)")
plt.show()

print("Sped up — half the samples, played at the same rate:")
display(Audio(fast_audio, rate=sr))

In [ ]:
# --- Add noise ---
noise = np.random.randn(len(audio)).astype(np.float32) * 0.05
noisy_audio = audio + noise

plt.figure(figsize=(12, 3))
librosa.display.waveshow(noisy_audio, sr=sr)
plt.title("Birdsong + Random Noise")
plt.show()

print("With added noise — sounds like a bad recording:")
display(Audio(noisy_audio, rate=sr))

In [ ]:
# --- Clip a section (just the first second) ---
one_second = audio[:sr]  # sr samples = 1 second

fig, axes = plt.subplots(1, 2, figsize=(14, 3))

librosa.display.waveshow(one_second, sr=sr, ax=axes[0])
axes[0].set_title("First Second — Waveform")

S = np.abs(librosa.stft(one_second))
S_db = librosa.amplitude_to_db(S, ref=np.max)
librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='log', ax=axes[1])
axes[1].set_title("First Second — Spectrogram")

plt.tight_layout()
plt.show()

print("Just the first second:")
display(Audio(one_second, rate=sr))

---
## What You Learned

| Concept | Plain English | What it looks like |
|---------|--------------|--------------------|
| **Digital audio** | A list of numbers measuring air pressure | `[0.02, -0.01, 0.03, ...]` |
| **Sample rate** | How many measurements per second | 22,050 Hz = 22,050 numbers/sec |
| **Waveform** | Those numbers drawn as a line | Loud = tall peaks, silent = flat |
| **Spectrogram** | A picture that pulls frequencies apart | Horizontal line = steady note, diagonal = sweep |
| **Manipulating audio** | Change the numbers, change the sound | Reverse, speed up, add noise, clip |

**Next step:** Open **Notebook 2 — Birdsong Explorer** to compare songs across five different species!